# 01 — Retrieval Analysis

This notebook walks through the retrieval evaluation pipeline and visualizes Hit@K, MRR, and failure clusters from the latest `retrieval_eval_seed_report.json`.

**Prereqs**: `app/evaluation/reports/retrieval_eval_seed_report.json` must exist (run `python -m app.evaluation.scripts.evaluate_retrieval` if missing).

In [1]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if (REPO_ROOT / 'app').exists() is False:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from app.analytics.analyze_retrieval import analyze_retrieval_report, hit_at_k_curve, cluster_failures, load_retrieval_report
from app.analytics.visualizer import plot_hit_at_k_curve, plot_failure_case_heatmap, plot_response_time_distribution

REPORT_PATH = REPO_ROOT / 'app/evaluation/reports/retrieval_eval_seed_report.json'
print('Report exists:', REPORT_PATH.exists())

Report exists: True


## 1. Load and inspect raw retrieval report

In [2]:
raw = load_retrieval_report(REPORT_PATH)
print('Top keys:', list(raw.keys()))
print('Sample count:', len(raw.get('results', [])))
print('Summary:', json.dumps(raw.get('summary', {}), ensure_ascii=False, indent=2))

Top keys: ['dataset', 'summary', 'results']
Sample count: 11
Summary: {
  "sample_count": 11,
  "top_k": 3,
  "hit_rate": 1.0,
  "mean_recall": 1.0,
  "mrr": 1.0,
  "hits": 11,
  "misses": 0
}


## 2. Hit@K curve

In [3]:
analysis = analyze_retrieval_report(REPORT_PATH)
curve = analysis['hit_at_k_curve']
print('Hit@K:', curve)
fig = plot_hit_at_k_curve(curve, title='Retrieval Hit@K (seed dataset)')
fig

Hit@K: {1: 1.0, 3: 1.0, 5: 1.0, 10: 1.0}


<Figure size 600x400 with 1 Axes>

## 3. Failure case clusters

In [4]:
clusters = analysis['failure_clusters']
print('Failures by paper:', clusters['by_paper'])
print('Failures by section:', clusters['by_section'])

matrix = {}
for paper, sids in clusters['by_paper'].items():
    matrix.setdefault(paper, {})
    for sect, sids2 in clusters['by_section'].items():
        overlap = len(set(sids) & set(sids2))
        if overlap:
            matrix[paper][sect] = overlap
fig2 = plot_failure_case_heatmap(matrix, title='Retrieval Failures by Paper × Section')
fig2

Failures by paper: {}
Failures by section: {}


<Figure size 600x300 with 1 Axes>

## 4. Live retrieval latency from analytics events (if available)

In [5]:
events_path = REPO_ROOT / 'app/storage/analytics/events.jsonl'
if events_path.exists():
    retrieval_times = []
    for line in events_path.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        rec = json.loads(line)
        if rec.get('event_type') == 'qa' and rec.get('payload', {}).get('retrieval_time') is not None:
            retrieval_times.append(float(rec['payload']['retrieval_time']))
    print(f'Live retrieval events: {len(retrieval_times)}')
    plot_response_time_distribution(retrieval_times, title='Live Retrieval Latency').show() if retrieval_times else None
else:
    print('No live events yet; run QA against the API to populate.')

No live events yet; run QA against the API to populate.


## 5. Conclusions and recommendations

- Current seed dataset metrics are largely synthetic (stub answers placed at rank 1).
- Hard out-of-scope samples expose real misses → use them to gate any Phase 4 retrieval upgrade.
- Next step: re-run `evaluate_qa --use-live-pipeline` against the real `paper_qa.answer_question` to populate `events.jsonl` with non-degenerate metrics.